# 11A · Inside the Candlestick — The Order Book
### Financial Analytics — Module 11 · Lab 3

Module 4 taught you to read a candlestick from outside. This notebook opens it up. Every price you have ever seen — every close, every tick — is the output of one machine: the **limit order book**. Understand the book and you understand what "the price" actually is, where trading costs really come from, and why speed became an arms race.

We build a toy book from scratch. No library, ~40 lines, full understanding.

> 🛡️ **Bias check:** this is a constructed teaching simulation (seed 11) — no history, so no history biases. The bias to guard here is *realism*: our toy has no strategic players reacting to you. Real books fight back. Every conclusion below is a mechanism demonstration, not a market measurement.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(11)

---
## 1. The book: two queues facing each other

A **limit order** says "I'll buy 100 shares, but only at ₹99.50 or less" — it *waits* in the book. The book is just two sorted queues: **bids** (buyers, best/highest first) and **asks** (sellers, best/lowest first). The gap between the best bid and best ask is the **spread** — and nobody trades until someone crosses it.

In [ ]:
# A toy book around Rs 100: price levels and the quantity resting at each
bids = pd.DataFrame({"price": [99.95, 99.90, 99.85, 99.80, 99.75],
                     "qty":   [  400,   900,  1500,  2200,  3000]})
asks = pd.DataFrame({"price": [100.05, 100.10, 100.15, 100.20, 100.25],
                     "qty":   [   350,   800,  1400,  2100,  2900]})

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(bids["price"], -bids["qty"], height=0.035, color="#16A34A", label="bids (buyers waiting)")
ax.barh(asks["price"],  asks["qty"], height=0.035, color="#DC2626", label="asks (sellers waiting)")
ax.axhline(100.00, color="black", ls=":", lw=1)
ax.set_title("The order book: depth at each price. The gap in the middle IS the spread (Rs 0.10)",
             loc="left", fontweight="bold")
ax.set_xlabel("quantity (bids shown negative)"); ax.set_ylabel("price"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"Best bid {bids.price[0]} | best ask {asks.price[0]} | spread Rs {asks.price[0]-bids.price[0]:.2f} "
      f"({(asks.price[0]-bids.price[0])/100*10000:.0f} basis points)")

**"The price" is a fiction of convenience.** There are always *two* prices — one to buy at, one to sell at — and the quoted "last price" is merely the most recent crossing. The spread is the toll every round trip pays, and notice who collects it...

## 2. A market order walks the book: impact, live

A **market order** says "fill me NOW at whatever's there." Small orders take the best price. Big orders *eat through levels* — and pay more with every level consumed:

In [ ]:
def market_buy(asks, qty):
    """Walk a buy order up the ask queue; return average fill price and the levels consumed."""
    remaining, cost, fills = qty, 0.0, []
    for _, row in asks.iterrows():
        take = min(remaining, row["qty"])
        cost += take * row["price"]; fills.append((row["price"], take))
        remaining -= take
        if remaining <= 0: break
    return cost/qty, fills

for q in [200, 1_000, 4_000]:
    avg, fills = market_buy(asks, q)
    print(f"BUY {q:>5} shares -> avg fill Rs {avg:.3f}  (vs best ask 100.05: paid {10000*(avg-100.05)/100.05:5.1f} bps extra)  levels used: {len(fills)}")

**Market impact, measured.** The 200-share order pays the quote. The 4,000-share order pays materially more — not fees, not spread: *its own size moved the price*. This is why big institutions slice orders over hours (execution algorithms — the unglamorous 90% of real "algo trading"), and why every backtest that assumes fills at the printed price flatters itself. **File that away hard: 11B charges for it.**

## 3. The market maker: paid to stand in the middle

Who supplies those resting orders? Often a **market maker** — quoting both sides, earning the spread, bearing the risk that prices move while inventory sits. Simulate a naive one for 500 ticks:

In [ ]:
true_px = 100 + np.cumsum(rng.normal(0, 0.03, 500))     # the 'fair value' random walk
HALF_SPREAD = 0.05
cash, inv = 0.0, 0
inv_hist, pnl_hist = [], []

for t in range(500):
    # We quote around fair value; a random trader hits one side (or none)
    side = rng.choice(["buy_from_us", "sell_to_us", "none"], p=[0.35, 0.35, 0.30])
    if side == "buy_from_us":     # they buy at our ASK (fair + half-spread)
        cash += true_px[t] + HALF_SPREAD; inv -= 1
    elif side == "sell_to_us":    # they sell at our BID
        cash -= true_px[t] - HALF_SPREAD; inv += 1
    inv_hist.append(inv); pnl_hist.append(cash + inv*true_px[t])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(pnl_hist, color="#16A34A"); axes[0].set_title("Market-maker P&L: the spread drips in", fontsize=10)
axes[1].plot(inv_hist, color="#7C3AED"); axes[1].axhline(0, color="black", lw=0.7)
axes[1].set_title("...but INVENTORY wanders: the risk being paid for", fontsize=10)
plt.tight_layout(); plt.show()
print(f"Final P&L: Rs {pnl_hist[-1]:.1f} | worst inventory position: {min(inv_hist)} to {max(inv_hist)} shares")

Two honest readings: the spread income is real (P&L drips upward) — and the **inventory is the danger** (the wandering purple line is unhedged exposure; a sharp move against a loaded book erases weeks of spread). Real market makers spend most of their engineering on inventory control — skewing quotes to shed positions — and on *not being the slowest*:

## 4. Why speed became an arms race — in one experiment

In [ ]:
# Fair value jumps. Two market makers see it - one updates quotes in 1 tick, one in 5 ticks.
# During the laggard's stale window, fast traders buy its old (too-cheap) ask. Count the damage:
N_JUMPS = 200
jump_sizes = rng.normal(0, 0.30, N_JUMPS)
stale_loss_slow = np.abs(jump_sizes[np.abs(jump_sizes) > 0.05]).sum() * 4   # 4 extra stale ticks x 1 lot
stale_loss_fast = np.abs(jump_sizes[np.abs(jump_sizes) > 0.05]).sum() * 0
print(f"Losses to stale quotes over {N_JUMPS} jumps: slow maker Rs {stale_loss_slow:.0f} | fast maker Rs {stale_loss_fast:.0f}")
print()
print("Being slow doesn't mean earning less - it means being everyone else's free option.")
print("THAT asymmetry - stale quotes are money lying on the floor - is the entire economic logic of the")
print("HFT arms race: microwave towers, co-located servers, nanosecond timestamps. Not glamour: self-defence.")

### The vocabulary you now own
**Limit vs market order** · **spread** (the toll) · **depth** (how much rests at each level) · **market impact** (your size moves your price) · **market making** (paid the spread, bears the inventory) · **latency** (stale quotes = free options for the fast) · **execution algorithms** (slicing big orders — most of real-world "algo trading" is this, not prediction).

### ✏️ Exercises
1. **The sell side:** write `market_sell(bids, qty)` and price a 4,000-share SELL. Is the impact symmetric with the buy side in our toy? What would make it asymmetric in a real panic?
2. **Wider spreads, calmer inventory:** re-run the market maker with HALF_SPREAD = 0.15 and hit-probabilities dropping to 0.2/0.2/0.6 (wider quotes attract less flow). More profit or less? What's the trade-off a real maker tunes all day?
3. **The inventory limiter:** modify the maker so that when |inventory| > 10, it skews — stops quoting the side that would grow the position. Compare the inventory band and final P&L. You just built the first risk control of every real trading system.

---
*AI disclosure: ______*

In [ ]:
# workspace
